<a href="https://colab.research.google.com/github/bsrikanth24/Best-websites-a-programmer-should-visit/blob/master/How_to_get_all_occurrences_of_duplicate_records_in_a_PySpark_DataFrame_based_on_specific_columns%3F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.appName("DataFrameExample").getOrCreate()

# Prepare Data
data = [("A", "A", 1), \
    ("A", "A", 2), \
    ("A", "A", 3), \
    ("A", "B", 4), \
    ("A", "B", 5), \
    ("A", "C", 6), \
    ("A", "D", 7), \
    ("A", "E", 8), \
  ]

# Create DataFrame
columns= ["col_1", "col_2", "col_3"]
df = spark.createDataFrame(data = data, schema = columns)
df.printSchema()
df.show(truncate=False)

root
 |-- col_1: string (nullable = true)
 |-- col_2: string (nullable = true)
 |-- col_3: long (nullable = true)

+-----+-----+-----+
|col_1|col_2|col_3|
+-----+-----+-----+
|A    |A    |1    |
|A    |A    |2    |
|A    |A    |3    |
|A    |B    |4    |
|A    |B    |5    |
|A    |C    |6    |
|A    |D    |7    |
|A    |E    |8    |
+-----+-----+-----+



In [3]:
not_duplicate_records = df.groupBy('col_1', 'col_2').count().where('count = 1').drop('count')
not_duplicate_records.show()

+-----+-----+
|col_1|col_2|
+-----+-----+
|    A|    C|
|    A|    E|
|    A|    D|
+-----+-----+



In [6]:
duplicate_records = df.join(not_duplicate_records, on=['col_1', 'col_2'], how='left_anti')
duplicate_records.show()

+-----+-----+-----+
|col_1|col_2|col_3|
+-----+-----+-----+
|    A|    A|    1|
|    A|    A|    2|
|    A|    A|    3|
|    A|    B|    4|
|    A|    B|    5|
+-----+-----+-----+



In [7]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

windowSpec  = Window.partitionBy('col_1', 'col_2')

df.withColumn("primary_key_count",F.count("*").over(windowSpec)).filter(F.col("primary_key_count") > 1).drop("primary_key_count").show(truncate=False)

+-----+-----+-----+
|col_1|col_2|col_3|
+-----+-----+-----+
|A    |A    |1    |
|A    |A    |2    |
|A    |A    |3    |
|A    |B    |4    |
|A    |B    |5    |
+-----+-----+-----+

